In [1]:
from typing import TypedDict
import time

from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver


# ============================================================
# 1. DEFINE STATE
# ============================================================

class CrashState(TypedDict):
    input: str
    step1: str
    step2: str
    step3: str


# ============================================================
# 2. DEFINE NODES
# ============================================================

def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")

    return {
        "input": state["input"],
        "step1": "done"
    }


def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 started...")
    print("🛑 Press STOP/Interrupt now to simulate failure")

    # Simulate a long-running operation
    time.sleep(10)

    return {
        "step2": "done"
    }


def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")

    return {
        "step3": "done"
    }


# ============================================================
# 3. BUILD GRAPH
# ============================================================

builder = StateGraph(CrashState)

builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")

builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)


# ============================================================
# 4. ADD PERSISTENCE
# ============================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# ============================================================
# 5. CONFIGURATION
# ============================================================

config = {
    "configurable": {
        "thread_id": "thread-1"
    }
}


# ============================================================
# 6. FIRST RUN
# ============================================================

try:

    print("\n▶️ FIRST RUN")
    print("-------------------------")

    result = graph.invoke(
        {
            "input": "start"
        },
        config=config
    )

    print("\nFirst run completed:")
    print(result)


except KeyboardInterrupt:

    print("\n")
    print("❌ Graph interrupted!")
    print("💾 But previous checkpoint is still available.")


# ============================================================
# 7. CHECK SAVED STATE
# ============================================================

print("\n")
print("🔍 CHECKPOINTED STATE")
print("-------------------------")

state = graph.get_state(config)

print(state)


# ============================================================
# 8. RESUME SAME THREAD
# ============================================================

print("\n")
print("🔁 RESUMING GRAPH")
print("-------------------------")

try:

    final_state = graph.invoke(
        None,
        config=config
    )

    print("\n✅ FINAL STATE:")
    print(final_state)

except KeyboardInterrupt:

    print("❌ Interrupted again!")


# ============================================================
# 9. SHOW STATE HISTORY
# ============================================================

print("\n")
print("📜 STATE HISTORY")
print("-------------------------")

history = list(
    graph.get_state_history(config)
)

for i, checkpoint in enumerate(history, 1):

    print(f"\nCheckpoint {i}:")
    print(checkpoint)


▶️ FIRST RUN
-------------------------
✅ Step 1 executed
⏳ Step 2 started...
🛑 Press STOP/Interrupt now to simulate failure
✅ Step 3 executed

First run completed:
{'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


🔍 CHECKPOINTED STATE
-------------------------
StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ad234-2cd6-6c93-8003-611029f18a01'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-09-10T14:23:58.141504+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1ad234-2ccd-64a4-8002-d26c16a1a655'}}, tasks=(), interrupts=())


🔁 RESUMING GRAPH
-------------------------

✅ FINAL STATE:
{'input': 'start', 'step1': 'done', 'step2': 'done', 'step3': 'done'}


📜 STATE HISTORY
-------------------------

Checkpoint 1:
StateSnapshot(values={'input': 